In [1]:
import os
import duckdb
from pathlib import Path
from dotenv import load_dotenv

In [2]:
notebook_dir = Path.cwd()
repo_root = notebook_dir.parent
load_dotenv(repo_root / ".env")

con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH", "data/f1_local.duckdb")))

In [3]:
con.sql("CREATE SCHEMA IF NOT EXISTS bronze")
con.sql("CREATE SCHEMA IF NOT EXISTS silver")
con.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [4]:
schemas = con.sql("SELECT schema_name FROM information_schema.schemata").fetchall()
print(schemas)
con.close()

[('bronze',), ('gold',), ('main',), ('silver',), ('information_schema',), ('main',), ('pg_catalog',), ('main',)]


In [2]:
repo_root = Path.cwd().parent
load_dotenv(repo_root / ".env")

con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH")))

In [3]:
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    SET s3_access_key_id = '{os.getenv("AWS_ACCESS_KEY_ID")}';
    SET s3_secret_access_key = '{os.getenv("AWS_SECRET_ACCESS_KEY")}';
    SET s3_region = '{os.getenv("AWS_REGION")}';
""")
bucket = os.getenv("S3_BUCKET_RAW")
print("DuckDB connected to S3 success")

DuckDB connected to S3 success


In [ ]:
checks = {
    "race_results": {"expected_rounds": 2, "key_col": "position"},
    "driver_standings": {"expected_rounds": 1, "key_col": "points"},
    "constructor_standings": {"expected_rounds": 1, "key_col": "points"},
    "race_schedule": {"expected_rounds": 22, "key_col": "race_date"},
}

for endpoint, dict in checks.items():
    path = f"s3://{bucket}/jolpica/2026/{endpoint}.parquet"

    row_count = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{path}')"
    ).fetchone()[0]

    round_count = con.execute(
        f"SELECT COUNT(DISTINCT round) FROM read_parquet('{path}')"
    ).fetchone()[0]

    null_count = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{path}') "
        f"WHERE {dict['key_col']} IS NULL"
    ).fetchone()[0]

    status = "✓" if round_count == dict["expected_rounds"] and null_count == 0 else "✗"
    print(
        f"{status} {endpoint:<25} "
        f"rows={row_count:<6} "
        f"rounds={round_count}/22   "
        f"nulls_on_{dict['key_col']}={null_count}"
    )

✓ race_results              rows=44     rounds=2/22   nulls_on_position=0
✓ driver_standings          rows=22     rounds=1/22   nulls_on_points=0
✓ constructor_standings     rows=11     rounds=1/22   nulls_on_points=0
✓ race_schedule             rows=22     rounds=22/22   nulls_on_race_date=0


In [12]:
path = f"s3://{bucket}/jolpica/2026/driver_standings.parquet"

result = con.execute(f"""
    SELECT driver_name, points, wins
    FROM read_parquet('{path}')
    WHERE round = 2
    ORDER BY position
    LIMIT 3
""").df()
print(result.to_string(index=False))

          driver_name  points  wins
       George Russell    51.0     1
Andrea Kimi Antonelli    47.0     1
      Charles Leclerc    34.0     0


In [3]:
import sys, os, duckdb
from pathlib import Path
from dotenv import load_dotenv
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))
load_dotenv(repo_root / ".env")
from ingestion.jolpica.bronze_loader import load_jolpica_bronze, verify_bronze_tables

In [4]:
bucket = os.getenv("S3_BUCKET_RAW")

In [6]:
con = duckdb.connect(str(repo_root / os.getenv("DUCKDB_PATH")))
load_jolpica_bronze(con, year=2026)
verify_bronze_tables(con)

bronze.jolpica_race_results              rows=44     rounds=2
bronze.jolpica_driver_standings          rows=22     rounds=1
bronze.jolpica_constructor_standings     rows=11     rounds=1
bronze.jolpica_race_schedule             rows=22     rounds=22
bronze.jolpica_pit_stops                 rows=51     rounds=2
